In [2]:
import pandas as pd
import numpy as np
import random
import joblib
import requests
import json
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Model

# -------------------- Load Models and Tools --------------------
print("📦 Loading models and encoders...")
lstm_model = load_model("models/lstm_model.h5")
xgb_model = joblib.load("models/xgb_model.pkl")
scaler = joblib.load("models/scaler.pkl")
label_encoder = joblib.load("models/label_encoder.pkl")

# -------------------- Load and Preprocess Data --------------------
print("📑 Loading data...")
df = pd.read_csv("gpin_data.csv")

# Extract columns
# Extract only feature columns (safely exclude target if it exists)
feature_cols = [col for col in df.columns if col not in ('target', 'target_encoded')]
X = df[feature_cols].values
X_scaled = scaler.transform(X)

# -------------------- Sample a Sequence --------------------
SEQ_LEN = 10
num_sequences = len(X_scaled) // SEQ_LEN
rand_idx = random.randint(0, num_sequences - 1)

# Slice sequence
sample_seq = X_scaled[rand_idx * SEQ_LEN: (rand_idx + 1) * SEQ_LEN]
X_sample = sample_seq.reshape(1, SEQ_LEN, len(feature_cols))

# -------------------- Extract Latent Features via LSTM --------------------
feature_extractor = Model(inputs=lstm_model.input, outputs=lstm_model.layers[-2].output)
latent_vector = feature_extractor.predict(X_sample)

# -------------------- Predict Attack Type via XGBoost --------------------
pred_numeric = xgb_model.predict(latent_vector)[0]
pred_label = label_encoder.inverse_transform([pred_numeric])[0]

print(f"\n🧪 Prediction: {pred_label}")

# -------------------- If Attack, Send to Gemma --------------------
def send_to_gemma(prompt):
    try:
        url = "http://127.0.0.1:11434/api/chat"
        headers = {"Content-Type": "application/json"}
        data = {
            "model": "gemma3",
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        }
        response = requests.post(url, headers=headers, data=json.dumps(data))
        res = response.json()

        gemma_response = res.get("message", {}).get("content", "")
        print(f"\n🧠 GEMMA’s Recommendation:\n{gemma_response}\n")

        with open("gpin_attack_logs.txt", "a", encoding="utf-8") as f:
            f.write("\n\n========== NEW ATTACK DETECTED ==========" + "\n")
            f.write(f"{prompt}\n\n")
            f.write("🧠 GEMMA Suggestion:\n")
            f.write(gemma_response + "\n")

    except Exception as e:
        print(f"⚠️ Error querying Gemma: {e}")

# If malicious packet, trigger prompt
if pred_label != "BENIGN":
    flow_features = sample_seq[-1]  # last packet of the sequence
    feature_summary = "\n".join([
        f"- {col}: {round(val, 4)}"
        for col, val in zip(feature_cols, flow_features)
    ])
    prompt = f"""
🚨 [ATTACK DETECTED]: {pred_label}

The model has detected a packet flow that appears malicious.

Behavioral indicators:
{feature_summary}

Suggest 5 ways to prevent or mitigate it from a cybersecurity and system-level perspective in brief, personalize it according to the packet features above.
"""
    send_to_gemma(prompt)
else:
    print("✅ No attack detected. Safe packet.")

📦 Loading models and encoders...


📑 Loading data...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 364ms/step

🧪 Prediction: IP_SPOOF

🧠 GEMMA’s Recommendation:
Okay, here are 5 ways to prevent or mitigate the detected "IP_SPOOF" attack, tailored to the provided packet flow data, presented from both a cybersecurity and system-level perspective:

**1. Enhanced SYN Flood Mitigation (Cybersecurity Focus - Immediate Action):**

* **Action:** Immediately increase the SYN timeout value on the firewall/IPS. The `syn_flag_cnt: 1.0623` suggests a potential SYN flood.  A shorter timeout (e.g., 60-120 seconds) will quickly drop connections from attackers attempting to exhaust server resources.
* **Rationale:** The elevated SYN flag count strongly indicates an attacker attempting to overwhelm the system with connection requests.


**2. Deep Packet Inspection (DPI) with IP Reputation (Cybersecurity & System):**

* **Action:** Implement or strengthen DPI with IP reputation checks.  The `IP_SPOOF` nature suggests the source IP is likely malicious.  The